In [1]:
import yfinance as yf
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import time

C:\Users\schak\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\schak\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ---------------- CONFIG ---------------- #

DB_URL = (
    "mssql+pyodbc://@localhost\\SQLEXPRESS/StockAnalystTrack"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

DEFAULT_LOOKBACK = 365

SLEEP = 1.5          # seconds between calls
MAX_RETRY = 3
BACKOFF = 10    


# ---------------- DB ---------------- #

engine = create_engine(DB_URL)


# ---------------- HELPERS ---------------- #

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def get_symbols():

    with engine.connect() as conn:

        df = pd.read_sql("""
            SELECT DISTINCT nse_ticker_yfinance
            FROM nseticker
            WHERE nse_ticker_yfinance IS NOT NULL
        """, conn)

    return df["nse_ticker_yfinance"].tolist()


def get_last_dates():

    with engine.connect() as conn:

        df = pd.read_sql("""
            SELECT symbol, MAX(date) AS last_date
            FROM dailyohlc
            GROUP BY symbol
        """, conn)

    df["last_date"] = pd.to_datetime(df["last_date"])

    return dict(zip(df["symbol"], df["last_date"]))

In [3]:
def fetch_yahoo(symbol, start, end):

    ticker = yf.Ticker(symbol)

    for attempt in range(MAX_RETRY):

        try:

            # Force metadata load
            _ = ticker.fast_info

            df = ticker.history(
                start=start,
                end=end,
                auto_adjust=False
            )

            if df.empty:
                raise ValueError("Empty data")

            return df


        except Exception as e:

            if attempt < MAX_RETRY - 1:

                wait = BACKOFF * (attempt + 1)

                print(f"   ⏳ Retry in {wait}s ({e})")

                time.sleep(wait)

            else:
                raise e

In [4]:
today = pd.Timestamp.today().date()

symbols = get_symbols()
last_dates = get_last_dates()

total = len(symbols)

print(f"📊 Symbols to process: {total}\n")


inserted = 0
failed = []

📊 Symbols to process: 213



In [ ]:
with engine.begin() as conn:

    for i, sym in enumerate(symbols, start=1):

        print(f"[{i}/{total}] {sym}")

        # Decide start date
        if sym in last_dates and pd.notna(last_dates[sym]):

            start = last_dates[sym].date() + timedelta(days=1)

        else:
            start = today - timedelta(days=DEFAULT_LOOKBACK)


        if start > today:

            print("   ✔ Up to date\n")
            continue


        start_s = start.strftime("%Y-%m-%d")
        end_s = today.strftime("%Y-%m-%d")


        try:

            print(f"   ⬇ Fetching {start_s} → {end_s}")

            df = fetch_yahoo(sym, start_s, end_s)

            df.reset_index(inplace=True)
            df["Date"] = df["Date"].dt.tz_localize(None)


            rows = []

            for _, r in df.iterrows():

                rows.append({
                    "symbol": sym,
                    "date": r["Date"],

                    "open": r["Open"],
                    "high": r["High"],
                    "low": r["Low"],
                    "close": r["Close"],

                    "volume": int(r["Volume"]),
                    "adjclose": r["Adj Close"]
                })


            if not rows:

                print("   ⚠ No new rows\n")
                continue


            conn.execute(
                text("""
                INSERT INTO dailyohlc
                (symbol, [date], [open], high, low, [close], volume, adjclose)
                VALUES
                (:symbol, :date, :open, :high, :low, :close, :volume, :adjclose)
                """),
                rows
            )


            inserted += len(rows)

            print(f"   ✅ Inserted {len(rows)} rows\n")


        except Exception as e:

            print(f"   ❌ Failed: {e}\n")
            failed.append(sym)


        time.sleep(SLEEP)


print("=========== SUMMARY ===========")
print(f"Inserted rows : {inserted}")
print(f"Failed stocks : {len(failed)}")

if failed:
l, m.n,b980u7hynbu gjvmhc6454 vcĀprint("Failed:", failed)  

print("===============================")

In [5]:
ticker = yf.Ticker("SBIN.NS")
x = ticker.fast_info

df = yf.download("SBIN.NS", period = "1y", interval="1d")

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [6]:
df

Price,Close,High,Low,Open,Volume
Ticker,SBIN.NS,SBIN.NS,SBIN.NS,SBIN.NS,SBIN.NS
Date,,,,,
2025-02-20,715.336365,717.100921,709.111324,710.630861,7423268
2025-02-21,707.787903,717.296977,705.827272,712.983564,6571602
2025-02-24,702.298157,705.582190,696.514270,696.514270,6912148
2025-02-25,696.906433,704.797962,696.024125,704.797962,8599635
2025-02-27,690.044250,703.572595,687.299319,701.906047,12033139
...,...,...,...,...,...
2026-02-16,1208.099976,1212.300049,1185.300049,1194.000000,14972739
2026-02-17,1213.400024,1225.500000,1204.099976,1206.099976,20107196


In [7]:
data = ticker.history(start="2025-10-01",end="2025-12-31",auto_adjust=False)

In [8]:
data

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2025-10-01 00:00:00+05:30,872.299988,876.450012,862.799988,864.099976,864.099976,10832384,0.0,0.0
2025-10-03 00:00:00+05:30,865.150024,873.000000,863.799988,867.299988,867.299988,8019442,0.0,0.0
2025-10-06 00:00:00+05:30,868.000000,875.299988,862.799988,874.049988,874.049988,7820352,0.0,0.0
2025-10-07 00:00:00+05:30,874.150024,877.000000,863.000000,864.700012,864.700012,9242580,0.0,0.0
2025-10-08 00:00:00+05:30,866.799988,867.799988,857.250000,858.250000,858.250000,7156112,0.0,0.0
...,...,...,...,...,...,...,...,...
2025-12-23 00:00:00+05:30,976.299988,977.549988,970.700012,971.900024,971.900024,5123287,0.0,0.0
2025-12-24 00:00:00+05:30,973.200012,977.299988,967.500000,968.950012,968.950012,4105643,0.0,0.0
2025-12-26 00:00:00+05:30,968.950012,971.000000,964.750000,966.299988,966.299988,3286321,0.0,0.0
